## **Importing Necessary Libraries**

In [ ]:
import os
import torch
import evaluate
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, TrainingArguments, Trainer

In [3]:
df = pd.read_csv(r"E:\Arshdeep CL NLP Project\Dataset\preprocessed\fake_news_cleaned.csv")
df = df.dropna(subset=["lemmatized_text", "label"])

## **Train-Test Split**

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["lemmatized_text"].tolist(), df["label"].tolist(), test_size=0.2, random_state=42
)

In [5]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)

# **Creating the News Dataset Class**

In [6]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)

# **Specifying the Model**

In [7]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=p.label_ids)["accuracy"],
        "f1": f1.compute(predictions=preds, references=p.label_ids)["f1"]
    }

## **Setting-Up the Training Arguments**

In [ ]:
training_args = TrainingArguments(
    output_dir="E:\Arshdeep CL NLP Project\models\distilbert_finetuned",
    do_train=True,
    do_eval=True,
    eval_steps=500,         # this defines, how often to run evaluation
    save_steps=500,         # this defines, how often to checkpoint
    save_total_limit=1,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

### **Start the Fine-Tuning⚙️**

In [17]:
trainer.train()

Step,Training Loss
500,0.013800
1000,0.008200
1500,0.005700
2000,0.007100
2500,0.004400
3000,0.001400
3500,0.003200
4000,0.003900
4500,0.002900
5000,0.000100


TrainOutput(global_step=6630, training_loss=0.003850679244525832, metrics={'train_runtime': 2036.9768, 'train_samples_per_second': 52.055, 'train_steps_per_second': 3.255, 'total_flos': 1.404618061648896e+16, 'train_loss': 0.003850679244525832, 'epoch': 3.0})

# **Saving the Fine-Tuned Model and Tokenizer**

In [18]:
model.save_pretrained(r"E:\Arshdeep CL NLP Project\models\distilbert_finetuned")
tokenizer.save_pretrained(r"E:\Arshdeep CL NLP Project\models\distilbert_finetuned")

('E:\\Arshdeep CL NLP Project\\models\\distilbert_finetuned\\tokenizer_config.json',
 'E:\\Arshdeep CL NLP Project\\models\\distilbert_finetuned\\special_tokens_map.json',
 'E:\\Arshdeep CL NLP Project\\models\\distilbert_finetuned\\vocab.txt',
 'E:\\Arshdeep CL NLP Project\\models\\distilbert_finetuned\\added_tokens.json',
 'E:\\Arshdeep CL NLP Project\\models\\distilbert_finetuned\\tokenizer.json')